In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import f1_score

## 1. Setting up Reproducibility and CUDA

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 2. Experiment configuration

In [3]:
NUM_CLASSES = 1102
IMAGE_SIZE = 224

SMOKE_TEST = True

FULL_EPOCHS = 6
SMOKE_EPOCHS = 1
EPOCHS = SMOKE_EPOCHS if SMOKE_TEST else FULL_EPOCHS

BATCH_SIZE = 64
NUM_WORKERS = 0

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 3
MIN_IMPROVEMENT = 1e-4

SMOKE_TRAIN_SAMPLES = 2048
SMOKE_VAL_SAMPLES = 512
SMOKE_TEST_SAMPLES = 512

print("Smoke test:", SMOKE_TEST)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

Smoke test: True
Epochs: 1
Batch size: 64


## 3. Project Root Setup

In [4]:
TRAIN_CSV = PROJECT_ROOT / "reports/surveillance_make_model_extension/mixed_web_surveillance_train.csv"
MIXED_VAL_CSV = PROJECT_ROOT / "reports/surveillance_make_model_extension/mixed_web_surveillance_val.csv"
SURV_TEST_CSV = PROJECT_ROOT / "reports/surveillance_make_model_extension/surveillance_make_model_official_test.csv"

WEB_SPLIT_CSV = PROJECT_ROOT / "reports/final_dataset_preparation/final_make_model_split.csv"
CLASS_MAP_PATH = PROJECT_ROOT / "reports/final_dataset_preparation/final_make_model_class_to_id.json"

V1_CHECKPOINT = PROJECT_ROOT / "models/final_training/make_model/resnet50_final_best.pt"

MODEL_OUT_DIR = PROJECT_ROOT / "models/final_training/make_model_v2"
REPORT_OUT_DIR = PROJECT_ROOT / "reports/final_training/make_model_v2"

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = MODEL_OUT_DIR / "resnet50_mixed_surveillance_best.pt"
LAST_CHECKPOINT = MODEL_OUT_DIR / "resnet50_mixed_surveillance_last.pt"

required = [
    TRAIN_CSV, MIXED_VAL_CSV, SURV_TEST_CSV,
    WEB_SPLIT_CSV, CLASS_MAP_PATH, V1_CHECKPOINT
]

for p in required:
    print(p, "->", p.exists())
    if not p.exists():
        raise FileNotFoundError(p)

C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\mixed_web_surveillance_train.csv -> True
C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\mixed_web_surveillance_val.csv -> True
C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\surveillance_make_model_extension\surveillance_make_model_official_test.csv -> True
C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_make_model_split.csv -> True
C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_make_model_class_to_id.json -> True
C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\models\final_training\make_model\resnet50_final_best.pt -> True


## 4. Setting Autovision Shared Root

In [5]:
os.environ["AUTOVISION_SHARED_ROOT"] = (
    r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
    r"\capstone-vehicle-metadata"
)

## 5. Dataset roots

In [6]:
def env_path(name):
    value = os.getenv(name)
    return Path(value).expanduser().resolve() if value else None

def first_existing_dir(candidates, label):
    checked = []
    for p in candidates:
        if p is None:
            continue
        p = Path(p)
        checked.append(p)
        if p.exists() and p.is_dir():
            print(f"{label}: {p}")
            return p
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" +
        "\n".join(str(x) for x in checked)
    )

shared = env_path("AUTOVISION_SHARED_ROOT")

WEB_ROOT = first_existing_dir([
    env_path("AUTOVISION_WEB_ROOT"),
    PROJECT_ROOT / "data/compcars_kaggle/extracted",
    shared / "data/compcars_kaggle/extracted" if shared else None,
], "CompCars web root")

SURV_ROOT = first_existing_dir([
    env_path("AUTOVISION_SURV_ROOT"),
    PROJECT_ROOT / "data/compcars_sv/extracted/sv_data",
    shared / "data/compcars_sv/extracted/sv_data" if shared else None,
], "CompCars surveillance root")

DATASET_ROOTS = {
    "compcars_web": WEB_ROOT,
    "compcars_surveillance": SURV_ROOT,
}

CompCars web root: G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\compcars_kaggle\extracted
CompCars surveillance root: G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\compcars_sv\extracted\sv_data


## 6. Loading manifests

In [7]:
train_df = pd.read_csv(TRAIN_CSV)
mixed_val_df = pd.read_csv(MIXED_VAL_CSV)
surv_test_df = pd.read_csv(SURV_TEST_CSV)
web_split_df = pd.read_csv(WEB_SPLIT_CSV)

with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
    class_to_id = json.load(f)

assert len(class_to_id) == NUM_CLASSES

web_val_df = mixed_val_df[mixed_val_df["validation_domain"] == "compcars_web"].copy()
surv_val_df = mixed_val_df[mixed_val_df["validation_domain"] == "compcars_surveillance"].copy()
web_test_df = web_split_df[web_split_df["split"] == "test"].copy()

print("Mixed train:", len(train_df))
print("  web:", (train_df["training_source"] == "compcars_web").sum())
print("  surveillance:", (train_df["training_source"] == "compcars_surveillance").sum())

print("\nValidation")
print("  web:", len(web_val_df))
print("  surveillance:", len(surv_val_df))

print("\nUntouched tests")
print("  web:", len(web_test_df))
print("  surveillance:", len(surv_test_df))

C:\Users\elesh\AppData\Local\Temp\ipykernel_33456\2166427201.py:1: DtypeWarning: Columns (0: file_path, 1: model_name, 2: source_dataset, 3: source_file_path, 4: model, 5: official_split, 6: color) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_CSV)


Mixed train: 125201
  web: 97641
  surveillance: 27560

Validation
  web: 12229
  surveillance: 3066

Untouched tests
  web: 12229
  surveillance: 13113


## 7. Path Resolvers and Sample Checks

In [8]:
def resolve_image_path(row):
    key = str(row["dataset_key"])
    rel = str(row["relative_path"]).replace("\\", "/")
    if key not in DATASET_ROOTS:
        raise KeyError(f"Unknown dataset_key: {key}")
    return DATASET_ROOTS[key] / Path(rel)

for name, frame in {
    "train": train_df,
    "web_val": web_val_df,
    "surv_val": surv_val_df,
    "web_test": web_test_df,
    "surv_test": surv_test_df,
}.items():
    sample = frame.sample(min(5, len(frame)), random_state=SEED)
    missing = [str(resolve_image_path(r)) for _, r in sample.iterrows() if not resolve_image_path(r).exists()]
    print(name, "missing:", len(missing))
    if missing:
        print("Example:", missing[0])
        raise FileNotFoundError(missing[0])

print("Sample path checks passed.")

train missing: 0
web_val missing: 0
surv_val missing: 0
web_test missing: 0
surv_test missing: 0
Sample path checks passed.


## 8. Transforms

In [9]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.80, 1.0),
        ratio=(0.75, 1.33),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 9. Dataset and smoke subsets

In [18]:
class MakeModelDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = resolve_image_path(row)
        with Image.open(path) as img:
            image = img.convert("RGB")
        image = self.transform(image)
        return image, int(row["class_id"])

def subset(df, n):
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=SEED).reset_index(drop=True)

if SMOKE_TEST:
    train_run_df = subset(train_df, SMOKE_TRAIN_SAMPLES)
    web_val_run_df = subset(web_val_df, SMOKE_VAL_SAMPLES)
    surv_val_run_df = subset(surv_val_df, SMOKE_VAL_SAMPLES)
    web_test_run_df = subset(web_test_df, SMOKE_TEST_SAMPLES)
    surv_test_run_df = subset(surv_test_df, SMOKE_TEST_SAMPLES)
else:
    train_run_df = train_df.copy()
    web_val_run_df = web_val_df.copy()
    surv_val_run_df = surv_val_df.copy()
    web_test_run_df = web_test_df.copy()
    surv_test_run_df = surv_test_df.copy()

print("Run sizes:")
print(len(train_run_df), len(web_val_run_df), len(surv_val_run_df), len(web_test_run_df), len(surv_test_run_df))

Run sizes:
2048 512 512 512 512


## 10. Filtering Smoke Subsets

In [19]:
def keep_locally_available(df, name):
    before = len(df)

    exists_mask = df.apply(
        lambda row: resolve_image_path(row).exists(),
        axis=1,
    )

    filtered = df[exists_mask].reset_index(drop=True)

    missing = before - len(filtered)

    print(
        f"{name}: {len(filtered):,}/{before:,} available "
        f"({missing:,} missing)"
    )

    return filtered


if SMOKE_TEST:
    train_run_df = keep_locally_available(
        train_run_df,
        "train smoke"
    )

    web_val_run_df = keep_locally_available(
        web_val_run_df,
        "web val smoke"
    )

    surv_val_run_df = keep_locally_available(
        surv_val_run_df,
        "surveillance val smoke"
    )

    web_test_run_df = keep_locally_available(
        web_test_run_df,
        "web test smoke"
    )

    surv_test_run_df = keep_locally_available(
        surv_test_run_df,
        "surveillance test smoke"
    )

train smoke: 2,047/2,048 available (1 missing)
web val smoke: 511/512 available (1 missing)
surveillance val smoke: 512/512 available (0 missing)
web test smoke: 511/512 available (1 missing)
surveillance test smoke: 512/512 available (0 missing)


## 11. DataLoaders

In [20]:
def make_loader(df, transform, shuffle):
    ds = MakeModelDataset(df, transform)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )

train_loader = make_loader(train_run_df, TRAIN_TRANSFORM, True)
web_val_loader = make_loader(web_val_run_df, EVAL_TRANSFORM, False)
surv_val_loader = make_loader(surv_val_run_df, EVAL_TRANSFORM, False)
web_test_loader = make_loader(web_test_run_df, EVAL_TRANSFORM, False)
surv_test_loader = make_loader(surv_test_run_df, EVAL_TRANSFORM, False)

print("Train batches:", len(train_loader))

Train batches: 32


## 12. Load the existing final ResNet50 checkpoint

In [21]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

checkpoint = torch.load(
    V1_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint architecture:", checkpoint.get("architecture"))
print("Checkpoint classes:", checkpoint.get("num_classes"))
print("Checkpoint image size:", checkpoint.get("image_size"))
print("Checkpoint best epoch:", checkpoint.get("epoch"))

assert checkpoint.get("architecture") == "resnet50"
assert checkpoint.get("num_classes") == NUM_CLASSES

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(DEVICE)

print("V1 checkpoint loaded successfully.")

Checkpoint architecture: resnet50
Checkpoint classes: 1102
Checkpoint image size: 224
Checkpoint best epoch: 15
V1 checkpoint loaded successfully.


## 13. Loss, optimizer and AMP

In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("Optimizer: AdamW")
print("LR:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Optimizer: AdamW
LR: 1e-05
Weight decay: 0.0001


## 14. Training/evaluation helpers

In [23]:
def domain_metrics(y_true, y_pred, top5_correct):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    present_classes = np.unique(y_true)

    return {
        "accuracy": float((y_true == y_pred).mean()),
        "top5_accuracy": float(top5_correct / len(y_true)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=present_classes,
                average="macro",
                zero_division=0,
            )
        ),
        "classes_present": int(len(present_classes)),
    }

@torch.inference_mode()
def evaluate(model, loader, name):
    model.eval()

    total_loss = 0.0
    n = 0
    top5_correct = 0
    y_true, y_pred = [], []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        bs = targets.size(0)
        total_loss += loss.item() * bs
        n += bs

        pred = logits.argmax(dim=1)
        top5 = logits.topk(5, dim=1).indices

        top5_correct += top5.eq(targets.unsqueeze(1)).any(dim=1).sum().item()

        y_true.extend(targets.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())

    m = domain_metrics(y_true, y_pred, top5_correct)
    m["loss"] = total_loss / n
    m["samples"] = n
    m["domain"] = name
    return m

def train_one_epoch(model, loader, epoch):
    model.train()

    total_loss = 0.0
    correct = 0
    n = 0

    for batch_idx, (images, targets) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = targets.size(0)
        total_loss += loss.item() * bs
        correct += logits.argmax(dim=1).eq(targets).sum().item()
        n += bs

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch} | batch {batch_idx}/{len(loader)} | loss {loss.item():.4f}")

    return {
        "loss": total_loss / n,
        "accuracy": correct / n,
        "samples": n,
    }

## 15. Fine-tuning

In [25]:
history = []
best_score = -1.0
best_epoch = None
no_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print("\n" + "=" * 70)
    print(f"EPOCH {epoch}/{EPOCHS}")
    print("=" * 70)

    train_m = train_one_epoch(model, train_loader, epoch)
    web_m = evaluate(model, web_val_loader, "web_val")
    surv_m = evaluate(model, surv_val_loader, "surveillance_val")

    selection_score = (web_m["macro_f1"] + surv_m["macro_f1"]) / 2

    record = {
        "epoch": epoch,
        "train_loss": train_m["loss"],
        "train_accuracy": train_m["accuracy"],
        "web_val_loss": web_m["loss"],
        "web_val_accuracy": web_m["accuracy"],
        "web_val_top5_accuracy": web_m["top5_accuracy"],
        "web_val_macro_f1": web_m["macro_f1"],
        "surveillance_val_loss": surv_m["loss"],
        "surveillance_val_accuracy": surv_m["accuracy"],
        "surveillance_val_top5_accuracy": surv_m["top5_accuracy"],
        "surveillance_val_macro_f1": surv_m["macro_f1"],
        "selection_score": selection_score,
    }
    history.append(record)

    print("\nTRAIN:", train_m)
    print("WEB VAL:", web_m)
    print("SURVEILLANCE VAL:", surv_m)
    print("Selection score:", round(selection_score, 4))

    if selection_score > best_score + MIN_IMPROVEMENT:
        best_score = selection_score
        best_epoch = epoch
        no_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "architecture": "resnet50",
            "num_classes": NUM_CLASSES,
            "image_size": IMAGE_SIZE,
            "web_val_accuracy": web_m["accuracy"],
            "web_val_top5_accuracy": web_m["top5_accuracy"],
            "web_val_macro_f1": web_m["macro_f1"],
            "surveillance_val_accuracy": surv_m["accuracy"],
            "surveillance_val_top5_accuracy": surv_m["top5_accuracy"],
            "surveillance_val_macro_f1": surv_m["macro_f1"],
            "selection_score": selection_score,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "smoke_test": SMOKE_TEST,
        }, BEST_CHECKPOINT)

        print("Saved best checkpoint:", BEST_CHECKPOINT)
    else:
        no_improvement += 1
        print(f"No improvement: {no_improvement}/{EARLY_STOPPING_PATIENCE}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "architecture": "resnet50",
        "num_classes": NUM_CLASSES,
        "image_size": IMAGE_SIZE,
        "selection_score": selection_score,
        "seed": SEED,
        "smoke_test": SMOKE_TEST,
    }, LAST_CHECKPOINT)

    pd.DataFrame(history).to_csv(
        REPORT_OUT_DIR / "training_history.csv",
        index=False,
    )

    if not SMOKE_TEST and no_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping.")
        break

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best score:", best_score)


EPOCH 1/1
Epoch 1 | batch 0/32 | loss 1.1207

TRAIN: {'loss': 0.7147324006908046, 'accuracy': 0.8861748900830484, 'samples': 2047}
WEB VAL: {'accuracy': 0.8884540117416829, 'top5_accuracy': 0.9726027397260274, 'macro_f1': 0.8876174682144831, 'classes_present': 402, 'loss': 0.44636985758745973, 'samples': 511, 'domain': 'web_val'}
SURVEILLANCE VAL: {'accuracy': 0.552734375, 'top5_accuracy': 0.751953125, 'macro_f1': 0.5748259602033187, 'classes_present': 212, 'loss': 2.7441397607326508, 'samples': 512, 'domain': 'surveillance_val'}
Selection score: 0.7312
Saved best checkpoint: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\models\final_training\make_model_v2\resnet50_mixed_surveillance_best.pt

Training complete.
Best epoch: 1
Best score: 0.7312217142089008


## 16. Loading best checkpoint

In [26]:
best_ckpt = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

best_model = models.resnet50(weights=None)
best_model.fc = nn.Linear(best_model.fc.in_features, NUM_CLASSES)
best_model.load_state_dict(best_ckpt["model_state_dict"], strict=True)
best_model = best_model.to(DEVICE)
best_model.eval()

print("Loaded:", BEST_CHECKPOINT)
print("Best epoch:", best_ckpt["epoch"])
print("Selection score:", best_ckpt["selection_score"])

Loaded: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\models\final_training\make_model_v2\resnet50_mixed_surveillance_best.pt
Best epoch: 1
Selection score: 0.7312217142089008


## 17. Final evaluation on untouched test domains

In [27]:
web_test_metrics = evaluate(
    best_model,
    web_test_loader,
    "frozen_web_test",
)

surv_test_metrics = evaluate(
    best_model,
    surv_test_loader,
    "official_surveillance_test",
)

print("\nFROZEN WEB TEST")
print(json.dumps(web_test_metrics, indent=2))

print("\nOFFICIAL SURVEILLANCE TEST")
print(json.dumps(surv_test_metrics, indent=2))


FROZEN WEB TEST
{
  "accuracy": 0.8943248532289628,
  "top5_accuracy": 0.9726027397260274,
  "macro_f1": 0.8883439943141436,
  "classes_present": 402,
  "loss": 0.4148642508251093,
  "samples": 511,
  "domain": "frozen_web_test"
}

OFFICIAL SURVEILLANCE TEST
{
  "accuracy": 0.55078125,
  "top5_accuracy": 0.720703125,
  "macro_f1": 0.5371019664141081,
  "classes_present": 202,
  "loss": 2.664444863796234,
  "samples": 512,
  "domain": "official_surveillance_test"
}


## 18. Saving results

In [28]:
results = {
    "experiment": "resnet50_v2_mixed_web_surveillance",
    "architecture": "resnet50",
    "num_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "best_epoch": int(best_ckpt["epoch"]),
    "selection_score": float(best_ckpt["selection_score"]),
    "mixed_train_rows_full": int(len(train_df)),
    "web_train_rows_full": int((train_df["training_source"] == "compcars_web").sum()),
    "surveillance_train_rows_full": int((train_df["training_source"] == "compcars_surveillance").sum()),
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "smoke_test": SMOKE_TEST,
    "seed": SEED,
    "frozen_web_test": web_test_metrics,
    "official_surveillance_test": surv_test_metrics,
    "note": "Re-run confidence calibration before deployment.",
}

json_path = REPORT_OUT_DIR / "resnet50_v2_final_metrics.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

comparison = pd.DataFrame([
    {"domain": "frozen_web_test", **{k:v for k,v in web_test_metrics.items() if k != "domain"}},
    {"domain": "official_surveillance_test", **{k:v for k,v in surv_test_metrics.items() if k != "domain"}},
])

csv_path = REPORT_OUT_DIR / "resnet50_v2_test_comparison.csv"
comparison.to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)
display(comparison)

Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_training\make_model_v2\resnet50_v2_final_metrics.json
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_training\make_model_v2\resnet50_v2_test_comparison.csv


,domain,accuracy,top5_accuracy,macro_f1,classes_present,loss,samples
0,frozen_web_test,0.894325,0.972603,0.888344,402,0.414864,511
1,official_surveillance_test,0.550781,0.720703,0.537102,202,2.664445,512
